# 1 — Reformat: CFD → Brackish frames on the VM disk

**Question.** Does a ~3.4M-parameter CenterNet-style head on *frozen* DINOv3 ConvNeXt-B features detect fish competitively on the Brackish source of the Community Fish Detection Dataset (CFD), scored by the same `pycocotools` harness as the released RF-DETR baselines on the *identical* val images?

**Layout — nothing touches the laptop.**

| Location | Holds |
|---|---|
| `/content/` (VM SSD, ephemeral) | CFD metadata, the Brackish images (re-fetched each session — cheap on the LILA/GCS pipe) |
| Google Drive `frozen-trunk-detection/` | converted DINOv3 backbone weights (`weights/`), run dirs (`runs/<name>/` — `best.pt`, `history.json`, `curves.png`, `predictions.json`), `results/` (manifest, baseline predictions, `results.csv`, viz) |

**Do not put images on Drive** — per-file Drive API latency makes 12k JPEG reads crawl.

**Runtime.** T4/L4 is enough here: only the head trains and the backbone runs under `no_grad`, so the run is very likely CPU-bound on JPEG decode + augmentation. The throughput probe in `2_training` measures GPU utilisation before you spend on anything bigger. If the VM dies, re-run `1_reformat` (minutes) — every artefact of value is already on Drive.

**Rules written before the numbers** (also in [`docs/report.md`](docs/report.md)): never compare against the README's `.609` (different split — we re-run their weights instead); the baselines likely *saw* these val images, so their number is biased in their favour — our win is conservative, our loss is ambiguous and is reported as ambiguous; report trainable params **and** total inference FLOPs.

---

**This notebook (1 of 4)** turns the CFD master metadata into a per-source manifest, subsets the Brackish source on the published `is_train` split, and fetches its frames to the VM disk resized to long side 1024.

## 1 · Drive, paths, code

Mounts Drive, fixes the four paths every cell below uses, clones the branch and installs it editable. Safe to re-run: the clone is wiped and redone each time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, shutil, subprocess, pathlib
DRIVE = '/content/drive/MyDrive/frozen-trunk-detection'
for sub in ('weights', 'runs', 'results', 'results/manifest', 'results/viz'):
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
REPO = '/content/crop-counter'
DATA = '/content/data/brackish'
CFD  = '/content/cfd'
os.makedirs(CFD, exist_ok=True)
print(os.listdir(DRIVE))

%cd /content
!rm -rf crop-counter
!git clone --branch poc/detection-head --depth 1 https://github.com/InsightML/crop-counter.git
%cd /content/crop-counter
!git log --oneline -3
# torch/torchvision come with the Colab image. torchmetrics + termcolor are needed only so
# Meta's dinov3 hubconf imports (it pulls the segmentors); pycocotools for COCO AP; ijson to
# stream the 1.9M-record CFD metadata without json.load-ing it.
!pip install -q -e ".[detection]" ijson
# A running kernel does not re-read site-packages' .pth files, so the editable install is invisible
# to THIS process until restart (subprocess calls like `!python -m cropcounter.train` see it fine).
import sys, importlib
if '/content/crop-counter/src' not in sys.path:
    sys.path.insert(0, '/content/crop-counter/src')
importlib.invalidate_caches()
import cropcounter, torch
print('cropcounter', cropcounter.__file__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2 · CFD metadata → per-source manifest

47 MB zipped; the JSON inside is streamed (`ijson`), never `json.load`-ed. The manifest is the source for every subsetting decision in the memo (empty-image fraction, `is_train` balance, box-size percentiles, stride-4 centre-cell collision rate, licence).

In [ ]:
META = f'{CFD}/community_fish_detection_dataset.json.zip'
if not os.path.exists(META):
    !wget -q --show-progress -O {META} https://lilawildlife.blob.core.windows.net/lila-wildlife/community-fish-detection-dataset/community_fish_detection_dataset.json.zip
t0 = time.time()
!python -m cropcounter.cfd manifest --metadata {META} --out {CFD}/manifest
print(f'manifest in {time.time()-t0:.0f}s')
for f in os.listdir(f'{CFD}/manifest'):
    shutil.copy(f'{CFD}/manifest/{f}', f'{DRIVE}/results/manifest/{f}')

## 3 · Subset — all of Brackish, honouring the published `is_train` split

In [ ]:
import csv, json
# Resolve the exact `dataset` field string for Brackish from the manifest rather than hard-coding it.
rows = list(csv.DictReader(open(f'{CFD}/manifest/manifest.csv')))
name_col = next(c for c in rows[0].keys() if c.lower() in ('dataset', 'source', 'name'))
BRACKISH_SOURCE = next(r[name_col] for r in rows if 'rackish' in r[name_col])
print('Brackish source string:', repr(BRACKISH_SOURCE))
!rm -rf {DATA}
!python -m cropcounter.cfd subset --metadata {META} --out {DATA} --sources "{BRACKISH_SOURCE}" --train-cap 100000 --val-cap 100000 --seed 0
print(json.dumps(json.load(open(f'{DATA}/subset_summary.json')), indent=1)[:2000])
!wc -l {DATA}/download_list.txt

## 4 · Fetch images to the VM disk, resized to long side 1024 on write

Resize at fetch, not at train: (i) fairness — the released baselines run at 1024/640 and AP is acutely resolution-sensitive for small objects; (ii) train/val scale consistency; (iii) disk. Boxes and `width`/`height` are rescaled into `annotations.json`; the native-scale file is kept as `annotations.native.json`.

In [ ]:
t0 = time.time()
!python -m cropcounter.cfd fetch --subset {DATA} --max-side 1024 --workers 32 --mirror gcs
print(f'fetched in {time.time()-t0:.0f}s')
!du -sh {DATA}/train/images {DATA}/val/images
!ls {DATA}/train/images | wc -l; ls {DATA}/val/images | wc -l

## 5 · Next

`2_training.ipynb` — backbone metric-reproduction check, throughput probe, the frozen go/no-go run and the linear probe. Keep this VM alive: the frames just fetched are on its disk.